# Thai Health Analytics: สำรวจข้อมูลทีละขั้น

Notebook นี้เริ่มจากการอ่านไฟล์ CSV ตรวจโครงสร้างข้อมูล คำนวณอัตราต่อประชากร 100,000 คน และสร้างกราฟพื้นฐาน

## 1. เตรียมไลบรารีและตำแหน่งไฟล์

`pandas` ใช้จัดการตาราง ส่วน `matplotlib` และ `seaborn` ใช้สร้างกราฟ

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis import (
    add_incidence_rates, diagnosis_daily_summary, province_summary, region_summary
)
from src.charts import top_provinces_chart, diagnosis_trend_chart
from src.data_loader import load_diagnosis_data, load_hypertension_data

HYPERTENSION_PATH = PROJECT_ROOT / 'data' / 'raw' / 'hypertension_all.csv'
DIAGNOSIS_PATH = PROJECT_ROOT / 'data' / 'raw' / 'diag_code_2016.csv'

## 2. อ่านข้อมูล

ฟังก์ชัน Loader จะตรวจคอลัมน์และแปลงข้อมูลตัวเลขให้พร้อมใช้ ถ้าไฟล์ไม่ถูกต้องจะแจ้งข้อผิดพลาดที่อ่านเข้าใจได้

In [ ]:
hypertension = load_hypertension_data(HYPERTENSION_PATH)
diagnosis = load_diagnosis_data(DIAGNOSIS_PATH)

print('ข้อมูลความดันโลหิตสูง:', hypertension.shape)
print('ข้อมูลการวินิจฉัย:', diagnosis.shape)
hypertension.head()

## 3. ตรวจคุณภาพข้อมูลเบื้องต้น

ก่อนวิเคราะห์ควรดูชนิดข้อมูล จำนวนค่าว่าง และข้อมูลซ้ำเสมอ

In [ ]:
print(hypertension.dtypes)
print('\nค่าว่างแต่ละคอลัมน์')
print(hypertension.isna().sum())
print('\nจำนวนแถวซ้ำ:', hypertension.duplicated().sum())

## 4. คำนวณอัตราต่อประชากร 100,000 คน

จำนวนผู้ป่วยอย่างเดียวอาจทำให้จังหวัดใหญ่ดูสูงกว่าจังหวัดเล็ก จึงใช้อัตราต่อแสนคนเพื่อช่วยเปรียบเทียบ

In [ ]:
hypertension_with_rates = add_incidence_rates(hypertension)
hypertension_with_rates[[
    'จังหวัด', '2559', '2560', '2561',
    'อัตรา_2559', 'อัตรา_2560', 'อัตรา_2561'
]].head()

## 5. จังหวัดที่มีอัตราสูงสุด

ตัวอย่างนี้เลือกปี 2561 และแสดง 10 จังหวัดแรก

In [ ]:
top_provinces = province_summary(hypertension, '2561', top_n=10)
top_provinces

In [ ]:
top_provinces_chart(top_provinces, '2561')
plt.show()

## 6. สรุประดับเขตบริการสุขภาพ

ฟังก์ชันจะรวมผู้ป่วยและประชากรของจังหวัดในเขตก่อน แล้วจึงคำนวณอัตรา วิธีนี้ถูกต้องกว่าการนำอัตราของจังหวัดมาเฉลี่ย

In [ ]:
regions = region_summary(hypertension, '2561')
regions

## 7. สำรวจข้อมูลการวินิจฉัยรายวัน

แต่ละตัวอักษรเป็นหมวดรหัสการวินิจฉัย ตัวอย่างนี้ดูหมวด `I` ตลอดปี 2016

In [ ]:
diagnosis_summary = diagnosis_daily_summary(diagnosis, 'I')
diagnosis_trend_chart(diagnosis_summary, 'I')
plt.show()

## สิ่งที่ควรทำต่อ

1. ลองเปลี่ยนปีจาก 2561 เป็น 2559 หรือ 2560
2. ลองเปลี่ยนหมวดรหัสจาก I เป็น A, B หรือหมวดอื่น
3. เขียนข้อสังเกตจากกราฟ โดยแยกสิ่งที่เห็นออกจากการคาดเดาสาเหตุ
4. เปิด `app.py` เพื่อดูว่าการวิเคราะห์ชุดเดียวกันถูกนำไปทำ Dashboard อย่างไร